<a href="https://colab.research.google.com/github/kyle-woodward/bq-ee-vectorsearch/blob/main/Google_Satellite_Embeddings_BigQuery_Vector_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ELT Workflow for Google Satellite Embeddings-based Vector Search, using BigQuery
#### Author: Kyle Woodward
#### Last Modified: 24 November, 2025

In [ ]:
import ee
print(ee.__version__)
import geemap
from pprint import pprint
from google.cloud.bigquery import Client

1.5.24


In [ ]:
PROJECT = "your-cloud-project"
BQ_DATASET = "aef_embeddings"
BQ_TABLE = "aef_2024_1k"

# Set the credentials and project
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Make the new BQ dataset that'll store a new BQ table
!bq mk --project_id={PROJECT} {BQ_DATASET}

BigQuery error in mk operation: Dataset 'sig-outreach:gse_embeddings' already
exists.


### Earth Engine routine

Define a search area, grid it, and export satellite embeddings for each grid tile

In [ ]:
# Define your search area and grid it
gaul = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1")
va = gaul.filterMetadata('ADM1_NAME','equals','Virginia').first().geometry()
SCALE = 1000
grid = va.coveringGrid(proj='EPSG:4326',scale=SCALE) # tile-grid using SCALE-sized grid cells

In [ ]:
m = geemap.Map()
m.addLayer(va)
m.addLayer(grid)
m.setCenter(-76.17,36.83,10)
m

Map(center=[36.83, -76.17], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataG…

In [ ]:
# Load AEF Satellite Embeddings over search area and for a particular year.
aef = (ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
.filterDate('2024-01-01', '2025-01-01')
.filterBounds(va)
.mosaic())

# setup a nested mean reducer
def reduce_nested(img,
                   fc,
                   reducer:ee.Reducer,
                   scale:int,
                   crs:str,
                   crs_transform:ee.List,
                   best_effort:bool,
                   maxPixels:int,
                   tileScale:int
                   ):
    def reduce(f):
        reduced = img.reduceRegion(reducer,
                               f.geometry(),
                               scale,
                               crs,
                               crs_transform,
                               best_effort,
                               maxPixels,
                               tileScale
                               )
        return f.set(reduced)
    all_reduced = fc.map(reduce)
    return all_reduced


In [ ]:
# Average AEF Satellite Embeddings pixels to tile grid cells.
chunks = 10
step = round(1/chunks,2)
grids = grid.randomColumn()

start = 0.0
end = step
for chunk in range(chunks):
  print(start, end)
  grid_chunk = grids.filter(ee.Filter.And(ee.Filter.gte('random',start),ee.Filter.lt('random',end)))

  # if you are not needing to do any other harmonization with other gridded products, we can set the mean reducer's scale to our tile grid scale
  # taking advantage of AEF's built-in pyramids to get approximate mean result without aggregating pixels at native resolution
  # (should be roughly 1-2 AEF pixels overlapping each tile at the tile grid's scale)
  aef_patch_embed_mapRR = reduce_nested(aef,
                                        grid_chunk,
                                        reducer=ee.Reducer.mean(),
                                        scale=SCALE,
                                        crs='EPSG:4326',
                                        crs_transform=None,
                                        best_effort=True,
                                        maxPixels=1e9,
                                        tileScale=None
                                        )
  taskBQ = ee.batch.Export.table.toBigQuery(collection=aef_patch_embed_mapRR,
                                            description=f"aef_tile_means_bq_chunks_{start}_{end}",
                                            table=f'{PROJECT}.{BQ_DATASET}.{BQ_TABLE}',
                                            append=True)

  taskBQ.start()
  print("exporting", start, end)
  start = round(start+step,2)
  end = round(end+step,2)
  # break


0.0 0.1
exporting 0.0 0.1
0.1 0.2
exporting 0.1 0.2
0.2 0.3
exporting 0.2 0.3
0.3 0.4
exporting 0.3 0.4
0.4 0.5
exporting 0.4 0.5
0.5 0.6
exporting 0.5 0.6
0.6 0.7
exporting 0.6 0.7
0.7 0.8
exporting 0.7 0.8
0.8 0.9
exporting 0.8 0.9
0.9 1.0
exporting 0.9 1.0


### Go to lunch! We can monitor our Earth Engine export tasks [here](https://console.cloud.google.com/earth-engine/tasks) in the meantime

Expect each task to complete within 1-2 hours.

### BigQuery Routine

Once all Earth Engine tasks have completed we can proceed with BigQuery transformations.

In [ ]:
# Postprocess AEF Satellite Embedding grid table so that A[xx] columns (AEF bands) form a single ARRAY<FLOAT> embedding column
from google.cloud import bigquery

# Dynamically generate the list of A00 to A63 columns
a_columns = [f"A{i:02d}" for i in range(64)] # Generates A00, A01, ..., A63
array_columns_str = ", ".join(a_columns) # Joins them with commas

query = f"""
SELECT
  geo,
  ARRAY[{array_columns_str}] AS embedding
FROM
  `{PROJECT}`.{BQ_DATASET}.{BQ_TABLE}
WHERE
A00 IS NOT NULL
"""

# Run the query
result_table = f'{PROJECT}.{BQ_DATASET}.{BQ_TABLE}_formatted'
job_config = bigquery.QueryJobConfig(destination=result_table)
client = bigquery.Client(project=PROJECT)
job = client.query(query, job_config=job_config)
job.result()  # Wait for the job to complete


In [ ]:
# Check if the result_table exists
def table_exists(client, table_id):
    try:
        client.get_table(table_id)
        print(f"Table {table_id} exists.")
        return True
    except Exception as e:
        print(f"Table {table_id} does not exist. Error: {e}")
        return False

table_exists(client, result_table)

Table sig-outreach.gse_embeddings.gse_2024_1k_formatted exists.


True

In [ ]:
# Check the resulting table's schema and data
query = f"SELECT geo,embedding FROM `{result_table}` LIMIT 10"
query_job = client.query(query)
schema = query_job.result().schema
for field in schema:
    print(f"{field.name}: {field.field_type}")
for row in query_job:
    print(row.embedding)

geo: GEOGRAPHY
embedding: FLOAT
[-0.010396001537870049, -0.16000000000000003, -0.16000000000000003, -0.03844675124951941, 0.019930795847750864, -0.019930795847750864, 0.008858131487889272, 0.07535563244905807, 0.029773164167627833, -0.0629911572472126, 0.11909265667051133, -0.09842368319876971, -0.04822760476739715, -0.14173010380622836, 0.04822760476739715, 0.07972318339100345, 0.029773164167627833, -0.08882737408688965, -0.05536332179930796, -0.0629911572472126, -0.007443291041906958, 0.007443291041906958, -0.03844675124951941, 0.22145328719723184, -0.12456747404844293, -0.08421376393694734, -0.0015378700499807767, -0.2599000384467512, 0.07111111111111111, -0.07972318339100345, -0.24415224913494812, 0.2288965782391388, 0.07972318339100345, -0.06698961937716265, -0.16633602460592078, -0.07111111111111111, 0.07972318339100345, -0.11909265667051133, 0.06698961937716265, 0.14173010380622836, -0.13016532103037293, -0.11374086889657825, 0.31009611687812383, 0.010396001537870049, -0.0553633

Index table for vector search

In [ ]:
# Create a vector index
in_table = '.'.join(result_table.split(".")[1:]) # remove project-id from table ref
print(f'indexing {in_table} for vector search')
query = f"""
CREATE VECTOR INDEX my_index ON {in_table}(embedding)
OPTIONS(distance_type='COSINE', index_type='IVF', ivf_options='{{"num_lists": 1000}}');
"""

# Run the query to create the index
client = bigquery.Client(project=PROJECT)
job = client.query(query)
job.result()  # Wait for the job to complete

table_exists(client, result_table)

indexing gse_embeddings.gse_2024_1k_formatted for vector search
Table sig-outreach.gse_embeddings.gse_2024_1k_formatted exists.


True

### Test Vector Search in BigQuery

Create a test target table for vector search (one embedding record to search for similar records in the whole embedding table)


In [ ]:
# Test vector search!
result_table = result_table+"_test_target"
query = f"SELECT * FROM {in_table} LIMIT 1"

job_config = bigquery.QueryJobConfig(destination=result_table)
job = client.query(query,job_config=job_config)
job.result()  # Wait for the job to complete

table_exists(client, result_table)

Table sig-outreach.gse_embeddings.gse_2024_1k_formatted_test_target exists.


True

In [ ]:
import datetime
target_table = '.'.join(result_table.split(".")[1:])
print(target_table)
query = f"""
SELECT query.geo AS target_geo,
  base.geo AS base_geo,
  distance
FROM
  VECTOR_SEARCH(
    TABLE {in_table},
    'embedding',
    TABLE {target_table},
    top_k => 11,
    distance_type => 'COSINE',
    options => '{{"fraction_lists_to_search": 0.005}}')
ORDER BY distance
LIMIT 10
OFFSET 1;
"""

# Run the query
client = bigquery.Client(project=PROJECT)
search_result_table = f"{PROJECT}.{BQ_DATASET}.vector_search_results_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
job_config = bigquery.QueryJobConfig(destination=search_result_table)
job = client.query(query,job_config=job_config)
job.result()  # Wait for the job to complete

table_exists(client, search_result_table)

gse_embeddings.gse_2024_1k_formatted_test_target
Table sig-outreach.gse_embeddings.vector_search_results_20251124_180817 exists.


True

In [ ]:
query = f"SELECT * FROM `{search_result_table}` LIMIT 10"
query_job = client.query(query)

for row in query_job:
    print(row['distance'], row['base_geo'])

0.0016933490055179856 POLYGON((-79.5727678673072 37.0555054699303, -79.563784714466 37.0555054699303, -79.563784714466 37.0644886227715, -79.5727678673072 37.0644886227715, -79.5727678673072 37.0555054699303))
0.0017274785831529194 POLYGON((-79.653616242878 37.0465223170891, -79.6446330900368 37.0465223170891, -79.6446330900368 37.0555054699303, -79.653616242878 37.0555054699303, -79.653616242878 37.0465223170891))
0.0017599202417925008 POLYGON((-79.5997173258308 37.0465223170891, -79.5907341729896 37.0465223170891, -79.5907341729896 37.0555054699303, -79.5997173258308 37.0555054699303, -79.5997173258308 37.0465223170891))
0.0018120537801880054 POLYGON((-79.6446330900368 37.0465223170891, -79.6356499371956 37.0465223170891, -79.6356499371956 37.0555054699303, -79.6446330900368 37.0555054699303, -79.6446330900368 37.0465223170891))
0.0025168933061959775 POLYGON((-79.5907341729896 37.0285560114067, -79.5817510201484 37.0285560114067, -79.5817510201484 37.0375391642479, -79.5907341729896 